In [ ]:
# 0. Imports and configuration

import re
import html
import json
import unicodedata
import warnings
import sys
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

sys.path.insert(0, str(Path.cwd()))
from project_paths import DATA_DIR, CLEANED_DIR, rel

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

# configuration
CONFIG = {
    "data_root": DATA_DIR,
    "out_root": CLEANED_DIR,
    "salary_cap": 1_000_000,
    "semantic_threshold": 0.95,
    "semantic_max_block": 1500,
}

SUBDIRS = ["normalized", "master", "analysis", "quarantine", "reports"]

# Audit accumulator -> saved to reports/data_quality_audit.json at the end
AUDIT = {"row_counts": {}, "issues": {}, "scores": {}, "notes": []}


def ensure_dirs():
    """Create the cleaned_data/ folder structure."""
    for d in SUBDIRS:
        (CONFIG["out_root"] / d).mkdir(parents=True, exist_ok=True)


def step(msg, df=None):
    """Lightweight progress print with shape."""
    if df is not None:
        print(f"[STEP] {msg:<52} rows={len(df):>9,}  cols={df.shape[1]}")
    else:
        print(f"[STEP] {msg}")


ensure_dirs()
print("Output folders ready under:", rel(CONFIG["out_root"]))

## 1. Load all datasets

In [ ]:
RAW = {}
for f in sorted(CONFIG["data_root"].rglob("*.csv")):
    rel = f.relative_to(CONFIG["data_root"]).as_posix()
    RAW[rel] = pd.read_csv(f, low_memory=False)
    AUDIT["row_counts"][f"raw::{rel}"] = int(len(RAW[rel]))
    step(f"loaded {rel}", RAW[rel])

print(f"\nLoaded {len(RAW)} datasets:", list(RAW.keys()))

In [ ]:
# null-percentage profile for the main tables
for name in ["postings.csv", "jobs/salaries.csv", "mappings/industries.csv"]:
    df = RAW[name]
    nulls = (df.isna().mean() * 100).round(2)
    high = nulls[nulls > 0].sort_values(ascending=False)
    print(f"\n=== {name}  ({len(df):,} rows x {df.shape[1]} cols) | null% > 0 ===")
    print(high.to_string() if len(high) else "  (no missing values)")

## 2. Cleaning helper functions

In [ ]:
URL_RE = re.compile(r"http\S+|www\.\S+", re.IGNORECASE)
EMAIL_RE = re.compile(r"\b[\w.+-]+@[\w-]+\.[\w.-]+\b")
PHONE_RE = re.compile(r"(?:\+?\d{1,3}[-.\s]?)?(?:\(?\d{3}\)?[-.\s]?)\d{3}[-.\s]?\d{4}")
HTML_RE = re.compile(r"<[^>]+>")
JD_HEADER_RE = re.compile(r"^\s*job\s*description\s*[:\-]?\s*", re.IGNORECASE)
MULTISPACE_RE = re.compile(r"\s+")


def light_clean_text(text, drop_contact=False):
    """Light, meaning-preserving clean: keeps case + punctuation."""
    if not isinstance(text, str) or not text.strip():
        return np.nan
    text = html.unescape(text)
    text = unicodedata.normalize("NFKC", text).replace("\ufffd", " ")
    text = HTML_RE.sub(" ", text)
    text = JD_HEADER_RE.sub("", text)
    if drop_contact:
        text = URL_RE.sub(" ", text)
        text = EMAIL_RE.sub(" ", text)
        text = PHONE_RE.sub(" ", text)
    text = text.replace("\r", " ").replace("\n", " ").replace("\t", " ")
    text = MULTISPACE_RE.sub(" ", text).strip()
    return text if text else np.nan


def lower_clean(text):
    """used for location matching keys."""
    v = light_clean_text(text)
    return v.lower() if isinstance(v, str) else np.nan


def norm_category(v):
    """Normalize a categorical value to snake_case (e.g. 'Full-time' -> 'full_time')."""
    v = light_clean_text(v)
    if not isinstance(v, str):
        return np.nan
    v = re.sub(r"[^a-z0-9]+", "_", v.lower()).strip("_")
    return v if v else np.nan


TITLE_ABBR = {
    r"\bsr\.?\b": "senior",
    r"\bjr\.?\b": "junior",
    r"\bmgr\.?\b": "manager",
    r"\bassoc\.?\b": "associate",
    r"\bdir\.?\b": "director",
}


def norm_title(t):
    """Normalize a job title for matching/grouping (lowercase, expand abbreviations)."""
    t = light_clean_text(t)
    if not isinstance(t, str):
        return np.nan
    t = t.lower()
    for pat, rep in TITLE_ABBR.items():
        t = re.sub(pat, rep, t)
    t = re.sub(r"[^a-z0-9+#/.\s-]", " ", t) 
    t = MULTISPACE_RE.sub(" ", t).strip()
    return t if t else np.nan


def norm_skill(s):
    """Normalize a skill name to snake_case."""
    s = light_clean_text(s)
    if not isinstance(s, str):
        return np.nan
    s = s.lower().replace("&", " and ")
    s = re.sub(r"[^a-z0-9+#. /-]", " ", s)
    s = re.sub(r"[\s/]+", "_", s).strip("_")
    return s if s else np.nan


# dtype fixing 
INT_COLS   = ["job_id", "company_id", "zip_code", "fips", "views", "applies",
              "industry_id", "salary_id", "employee_count", "follower_count"]
EPOCH_COLS = ["original_listed_time", "listed_time", "expiry", "closed_time", "time_recorded"]


def fix_types(df):
    df = df.copy()
    for c in INT_COLS:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")
    for c in EPOCH_COLS:
        if c in df.columns:
            df[c + "_dt"] = pd.to_datetime(df[c], unit="ms", errors="coerce")
    return df


# quick self-test of the bug-fix
_demo = "Job descriptionA Senior Python (C++/.NET) Engineer\n\nhttp://x.com  a@b.com"
print("demo ->", light_clean_text(_demo, drop_contact=True))
print("skill ->", norm_skill("Art/Creative"), "|", norm_skill("Information Technology"))

## 3. Resolve mappings (abbreviations / codes -> full names)

In [ ]:
# 3a. Skills: resolve abbreviation -> full name, aggregate per job
skills_map = RAW["mappings/skills.csv"].copy()
skills_map["skill_name_clean"] = skills_map["skill_name"].map(norm_skill)
abbr_to_name      = dict(zip(skills_map["skill_abr"], skills_map["skill_name"]))
abbr_to_nameclean = dict(zip(skills_map["skill_abr"], skills_map["skill_name_clean"]))

job_skills = fix_types(RAW["jobs/job_skills.csv"])
job_skills["skill_name"]  = job_skills["skill_abr"].map(abbr_to_name)
job_skills["skill_clean"] = job_skills["skill_abr"].map(abbr_to_nameclean)

job_skills_agg = (
    job_skills.dropna(subset=["skill_clean"])
    .groupby("job_id")["skill_clean"]
    .apply(lambda s: sorted(set(s)))
    .reset_index(name="required_skills")
)
step("job_skills resolved & aggregated", job_skills_agg)


# 3b. Industries: resolve id -> full name (fill missing), aggregate per job
ind_map = RAW["mappings/industries.csv"].copy()
ind_map["industry_name"] = ind_map["industry_name"].fillna(
    "unknown_industry_" + ind_map["industry_id"].astype(str)
)
id_to_industry = dict(zip(ind_map["industry_id"], ind_map["industry_name"]))

job_ind = fix_types(RAW["jobs/job_industries.csv"])
job_ind["industry_name"] = job_ind["industry_id"].map(id_to_industry)
job_ind_agg = (
    job_ind.dropna(subset=["industry_name"])
    .groupby("job_id")["industry_name"]
    .apply(lambda s: sorted(set(s)))
    .reset_index(name="industries")
)
step("job_industries resolved & aggregated", job_ind_agg)


# 3c. Benefits: aggregate per job
benefits = fix_types(RAW["jobs/benefits.csv"])
benefits_agg = (
    benefits.dropna(subset=["type"])
    .groupby("job_id")["type"]
    .apply(lambda s: sorted(set(s)))
    .reset_index(name="benefits")
)
step("benefits aggregated", benefits_agg)

In [ ]:
# 3d. Company attributes -> company_master
comp_ind = fix_types(RAW["companies/company_industries.csv"])
comp_ind_agg = (
    comp_ind.dropna(subset=["industry"])
    .groupby("company_id")["industry"]
    .apply(lambda s: sorted(set(s)))
    .reset_index(name="industries")
)

comp_spec = fix_types(RAW["companies/company_specialities.csv"])
comp_spec_agg = (
    comp_spec.dropna(subset=["speciality"])
    .groupby("company_id")["speciality"]
    .apply(lambda s: sorted(set(s)))
    .reset_index(name="specialities")
)

# employee_counts: keep latest snapshot per company (by time_recorded)
emp = fix_types(RAW["companies/employee_counts.csv"])
emp_latest = (
    emp.sort_values("time_recorded")
    .groupby("company_id", as_index=False)
    .tail(1)[["company_id", "employee_count", "follower_count"]]
    .rename(columns={"employee_count": "employee_count_latest",
                     "follower_count": "follower_count_latest"})
)

companies = fix_types(RAW["companies/companies.csv"])
companies["company_description_clean"] = companies["description"].map(
    lambda x: light_clean_text(x, drop_contact=True)
)

company_master = (
    companies[["company_id", "name", "company_description_clean",
               "company_size", "city", "state", "country"]]
    .merge(comp_ind_agg, on="company_id", how="left")
    .merge(comp_spec_agg, on="company_id", how="left")
    .merge(emp_latest, on="company_id", how="left")
)
company_master["company_industry"] = company_master["industries"].map(
    lambda x: x[0] if isinstance(x, list) and x else np.nan
)
step("company_master built", company_master)

## 4. Clean `postings.csv`

In [ ]:
KEEP = ["job_id", "title", "company_name", "company_id", "location",
        "formatted_work_type", "formatted_experience_level", "pay_period", "currency",
        "min_salary", "max_salary", "med_salary", "normalized_salary",
        "remote_allowed", "listed_time", "expiry", "description"]

post = fix_types(RAW["postings.csv"][KEEP])

# text & categorical normalization
post["title_clean"] = post["title"].map(norm_title)
post["company_name_clean"] = post["company_name"].map(norm_category)
post["location_clean"] = post["location"].map(lower_clean)
post["experience_required"] = post["formatted_experience_level"].map(norm_category)
post["work_type"] = post["formatted_work_type"].map(norm_category)
post["pay_period_clean"] = post["pay_period"].map(norm_category)
post["description_clean"] = post["description"].map(lambda x: light_clean_text(x, drop_contact=True))
post = post.drop(columns=["description"])

# salary: flag missingness, null impossible outliers
SAL = ["min_salary", "max_salary", "med_salary", "normalized_salary"]
post["has_salary"] = post[SAL].notna().any(axis=1)

ns = pd.to_numeric(post["normalized_salary"], errors="coerce")
upper = CONFIG["salary_cap"]
post["normalized_salary_outlier"] = ns.notna() & ((ns <= 0) | (ns > upper))
post["normalized_salary_clean"] = ns.mask(post["normalized_salary_outlier"])
AUDIT["issues"]["salary_outliers_nulled"] = int(post["normalized_salary_outlier"].sum())
AUDIT["issues"]["salary_cap_used"] = float(upper)

# remote flag: 1.0 / NaN -> clear boolean
post["remote_allowed_bool"] = post["remote_allowed"].notna() & (post["remote_allowed"] == 1)

step("postings cleaned", post)
print("salary outliers nulled:", AUDIT['issues']['salary_outliers_nulled'],
      "| salary cap:", f"{upper:,.0f}")

## 5. Referential integrity + quarantine


In [ ]:
# Referential integrity: split orphans, quarantine them
valid_jobs = set(post["job_id"].dropna().astype("int64"))


def split_orphans(df, key, valid):
    keep_mask = df[key].astype("Int64").isin(valid)
    return df[keep_mask].copy(), df[~keep_mask].copy()


child_tables = {
    "salaries":       fix_types(RAW["jobs/salaries.csv"]),
    "job_skills":     job_skills,
    "job_industries": job_ind,
    "benefits":       benefits,
}

clean_children = {}
for name, df in child_tables.items():
    clean, orphan = split_orphans(df, "job_id", valid_jobs)
    clean_children[name] = clean
    AUDIT["issues"][f"orphans_{name}"] = int(len(orphan))
    if len(orphan):
        orphan.to_csv(CONFIG["out_root"] / "quarantine" / f"orphan_{name}.csv", index=False)
    step(f"{name}: kept {len(clean):,} / quarantined {len(orphan):,}")

print("\nQuarantine summary:", {k: v for k, v in AUDIT["issues"].items() if k.startswith("orphans_")})

## 6. Deduplication

In [ ]:
# 6. Deduplication
n0 = len(post)

# (1) exact duplicate description text
dup_text = post["description_clean"].duplicated(keep="first") & post["description_clean"].notna()
post = post[~dup_text].reset_index(drop=True)
AUDIT["issues"]["dup_exact_description"] = int(dup_text.sum())

# (2) partial duplicates (only when all 3 keys are present)
key_cols = ["title_clean", "company_name_clean", "location_clean"]
dup_partial = post.duplicated(subset=key_cols, keep="first") & post[key_cols].notna().all(axis=1)
post = post[~dup_partial].reset_index(drop=True)
AUDIT["issues"]["dup_partial"] = int(dup_partial.sum())

step(f"after exact+partial dedup (removed {n0 - len(post):,})", post)


# (3) semantic near-duplicates, blocked by company
def semantic_duplicate_pairs(df, text_col, id_col, block_col, threshold, max_block):
    rows = []
    sub = df[[id_col, block_col, text_col]].dropna(subset=[text_col])
    for _, g in sub.groupby(block_col):
        if not (2 <= len(g) <= max_block):
            continue
        vec = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
        try:
            X = vec.fit_transform(g[text_col].astype(str))   # l2-normalized -> dot = cosine
        except ValueError:
            continue
        sims = (X @ X.T).toarray()
        ids = g[id_col].tolist()
        for i in range(len(ids)):
            for j in range(i + 1, len(ids)):
                if sims[i, j] >= threshold:
                    rows.append((ids[i], ids[j], round(float(sims[i, j]), 4)))
    return pd.DataFrame(rows, columns=["job_id", "duplicate_id", "similarity"])


sem_pairs = semantic_duplicate_pairs(
    post, "description_clean", "job_id", "company_id",
    CONFIG["semantic_threshold"], CONFIG["semantic_max_block"],
)
sem_pairs.to_csv(CONFIG["out_root"] / "reports" / "semantic_duplicates.csv", index=False)

drop_ids = set(sem_pairs["duplicate_id"].tolist())
post = post[~post["job_id"].isin(drop_ids)].reset_index(drop=True)
AUDIT["issues"]["dup_semantic"] = int(len(drop_ids))

step(f"after semantic dedup (removed {len(drop_ids):,})", post)
AUDIT["row_counts"]["postings_deduped"] = int(len(post))

## 7. Build `job_master`

In [ ]:
comp_attr = company_master[["company_id", "company_industry",
                            "company_size", "employee_count_latest"]]

jm = (
    post
    .merge(job_skills_agg, on="job_id", how="left")
    .merge(job_ind_agg,   on="job_id", how="left")
    .merge(benefits_agg,  on="job_id", how="left")
    .merge(comp_attr,     on="company_id", how="left")
)

# feature columns
jm["required_skill_count"]  = jm["required_skills"].map(lambda x: len(x) if isinstance(x, list) else 0)
jm["description_word_count"] = jm["description_clean"].fillna("").str.split().str.len()
jm["jd_completeness_score"] = np.select(
    [jm["description_word_count"] < 50,
     jm["description_word_count"].between(50, 150),
     jm["description_word_count"] > 150],
    [0.25, 0.60, 1.00], default=0.50,
)
jm["job_quality_score"] = (
    0.6 * jm["jd_completeness_score"]
    + 0.4 * np.minimum(jm["required_skill_count"] / 5, 1)
).round(3)

job_master = jm[[
    "job_id", "title", "title_clean", "company_id", "company_name", "company_industry",
    "company_size", "employee_count_latest", "description_clean", "required_skills",
    "required_skill_count", "experience_required", "industries", "work_type",
    "location", "location_clean", "min_salary", "max_salary", "normalized_salary_clean",
    "has_salary", "pay_period_clean", "currency", "benefits", "remote_allowed_bool",
    "listed_time_dt", "expiry_dt", "description_word_count",
    "jd_completeness_score", "job_quality_score",
]].rename(columns={"pay_period_clean": "pay_period"})

step("job_master assembled", job_master)
job_master.head(3)

## 8. Skill analysis, validation, readiness scores, and save outputs

In [ ]:
all_skills = [s for lst in job_master["required_skills"] if isinstance(lst, list) for s in lst]
skill_freq = (
    pd.Series(Counter(all_skills)).sort_values(ascending=False)
    .rename_axis("skill").reset_index(name="job_count")
)
skill_freq.to_csv(CONFIG["out_root"] / "analysis" / "skill_frequency.csv", index=False)
skill_freq.head(50).to_csv(CONFIG["out_root"] / "analysis" / "top_skills.csv", index=False)
print("Top 10 skills:\n", skill_freq.head(10).to_string(index=False))


assert job_master["job_id"].is_unique, "job_id must be unique in job_master"
assert job_master["job_id"].notna().all(), "job_id must not be null"
print("\nValidation OK: job_master job_id is unique and non-null.")

cov = lambda c: round(float(job_master[c].notna().mean()) * 100, 1)
metrics = {
    "n_jobs": int(len(job_master)),
    "title_coverage_pct": cov("title_clean"),
    "description_coverage_pct": cov("description_clean"),
    "experience_coverage_pct": cov("experience_required"),
    "skill_coverage_pct": round(float((job_master["required_skill_count"] > 0).mean()) * 100, 1),
    "salary_coverage_pct": round(float(job_master["has_salary"].mean()) * 100, 1),
}


scores = {
    "data_quality": int(round(0.5 * metrics["description_coverage_pct"] + 0.5 * metrics["title_coverage_pct"])),
    "nlp_readiness": int(round(metrics["description_coverage_pct"])),
    "ml_readiness": int(round(0.6 * metrics["description_coverage_pct"] + 0.4 * metrics["skill_coverage_pct"])),
    "matching_readiness": int(round(0.4 * metrics["skill_coverage_pct"]
                                    + 0.3 * metrics["description_coverage_pct"]
                                    + 0.3 * metrics["experience_coverage_pct"])),
}
AUDIT["scores"] = scores
AUDIT["metrics"] = metrics
AUDIT["notes"] = [
    "Resume side deferred until resume dataset is provided.",
    "normalized_salary outliers nulled into normalized_salary_clean; raw kept in salaries_clean.",
    "Orphan child rows quarantined under cleaned_data/quarantine/.",
]
print("\nMetrics:", json.dumps(metrics, indent=2))
print("Scores :", json.dumps(scores, indent=2))

In [ ]:
def serialize_lists(df):
    """Convert list-valued columns to pipe-joined strings"""
    df = df.copy()
    for c in df.columns:
        if df[c].map(lambda v: isinstance(v, list)).any():
            df[c] = df[c].map(lambda v: "|".join(map(str, v)) if isinstance(v, list) else "")
    return df


# master tables
serialize_lists(job_master).to_csv(CONFIG["out_root"] / "master" / "job_master.csv", index=False)
serialize_lists(company_master).to_csv(CONFIG["out_root"] / "master" / "company_master.csv", index=False)

# normalized, mapping-resolved tables
normalized = {
    "postings_clean": post,
    "companies_clean": companies.drop(columns=["description"], errors="ignore"),
    "skills_map_clean": skills_map,
    "industries_map_clean": ind_map,
    "job_skills_resolved": clean_children["job_skills"][["job_id", "skill_abr", "skill_name", "skill_clean"]],
    "job_industries_resolved": clean_children["job_industries"][["job_id", "industry_id", "industry_name"]],
    "salaries_clean": clean_children["salaries"],
    "benefits_clean": clean_children["benefits"],
    "company_industries_clean": comp_ind,
    "company_specialities_clean": comp_spec,
    "employee_counts_latest": emp_latest,
}
for name, df in normalized.items():
    serialize_lists(df).to_csv(CONFIG["out_root"] / "normalized" / f"{name}.csv", index=False)

# audit report
AUDIT["row_counts"]["job_master"] = int(len(job_master))
AUDIT["row_counts"]["company_master"] = int(len(company_master))
with open(CONFIG["out_root"] / "reports" / "data_quality_audit.json", "w", encoding="utf-8") as fh:
    json.dump(AUDIT, fh, indent=2, default=str)

print("Saved master tables:", [p.name for p in (CONFIG['out_root'] / 'master').glob('*.csv')])
print("Saved normalized tables:", len(normalized))
print("\nDONE. Outputs in:", rel(CONFIG["out_root"]))